# PhishGaurd — Dataset Creation and Teacher Annotation

This notebook documents the complete pipeline for building the **996-sample teacher-supervised dataset** used to fine-tune Gemma-2-2B-IT.

## What this notebook does

**Part 1 — Dataset Construction (946 + 50 emails)**
1. Download and load the Kaggle `CEAS_08` phishing email dataset
2. Clean, deduplicate, and token-filter the emails
3. Sample 946 emails from the cleaned pool
4. Define 50 manually crafted phishing and safe email scenarios

**Part 2 — Teacher Annotation with Gemini 2.5 Pro**

5. Set up the Gemini API (teacher LLM)
6. Define the teacher instruction prompt that extracts structured explanations
7. Call Gemini 2.5 Pro for every email to generate label, explanation, evidence snippets, and user advice
8. Save outputs as a chat-format JSONL file
9. Validate and filter hallucinated evidence snippets
10. Produce the final clean JSONL ready for Gemma fine-tuning

---
**Why teacher annotation?**  
Rather than training the SLM on binary labels alone, we use a large teacher LLM (Gemini 2.5 Pro) to generate high-quality structured explanations for each email. The SLM then learns to produce both correct classifications *and* coherent human-readable reasoning in a single forward pass.

---
# Part 1 — Dataset Construction

## Step 0 — Imports and Seed

In [ ]:
import os
import re
import json
import time
import hashlib
import unicodedata
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd

# Fix random seed for reproducible sampling
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_colwidth", 200)

## Step 1 — Download the Kaggle Dataset

**Dataset:** `naserabdullahalam/phishing-email-dataset` on Kaggle  
We use `CEAS_08.csv` which contains labeled emails with `subject`, `body`, and `label` columns (0 = safe, 1 = phishing).

> **Setup:** Go to Kaggle → Account → Create New API Token → upload `kaggle.json` to Colab, then run the cell below.

In [ ]:
KAGGLE_DATASET = "naserabdullahalam/phishing-email-dataset"

data_dir = Path.cwd() / "kaggle_data"
data_dir.mkdir(parents=True, exist_ok=True)

!pip -q install kaggle

if not Path("kaggle.json").exists():
    raise FileNotFoundError("kaggle.json not found. Upload it to Colab first.")

os.makedirs(Path.home() / ".kaggle", exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d {KAGGLE_DATASET} -p {data_dir} --unzip

print("Files downloaded:", [p.name for p in data_dir.glob("*")])

## Step 2 — Load CEAS_08 and Select Columns

In [ ]:
df_raw = pd.read_csv("kaggle_data/CEAS_08.csv")
print("Raw shape:", df_raw.shape)

# Verify required columns exist
REQUIRED_COLS = {"subject", "body", "label"}
missing = REQUIRED_COLS - set(df_raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df_raw[["subject", "body", "label"]].copy()

# Fill missing subject/body with empty string
df["subject"] = df["subject"].fillna("").astype(str)
df["body"]    = df["body"].fillna("").astype(str)

# Coerce labels to int, drop any unparseable rows
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

# Confirm only 0 and 1 labels exist
unexpected = set(df["label"].unique()) - {0, 1}
if unexpected:
    raise ValueError(f"Unexpected label values: {unexpected}")

print("Shape after selection:", df.shape)
print(df["label"].value_counts())

## Step 3 — Normalize Newlines, Build `email_text`, and Clean Text

We combine subject and body into a single `email_text` field using clear section headers, then apply text cleaning to remove unicode noise, control characters, and excessive whitespace.

In [ ]:
def normalize_newlines(s: str) -> str:
    """Replace escaped newline strings with real newlines and cap consecutive blank lines."""
    s = s.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\r", "\n")
    s = re.sub(r"\n{4,}", "\n\n\n", s)
    return s.strip()

# Compiled regex patterns for cleaning
CTRL_RE    = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")  # control chars and null bytes
WS_RE      = re.compile(r"[ \t]{2,}")                           # runs of spaces/tabs
MANY_NL_RE = re.compile(r"\n{5,}")                              # 5+ consecutive blank lines

def clean_text_for_llm(s: str) -> str:
    """Full cleaning pass: normalize unicode, strip control chars, compress whitespace."""
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKC", s)          # normalize unicode (e.g. full-width chars)
    s = s.replace("\r\n", "\n").replace("\r", "\n")  # unify line endings
    s = CTRL_RE.sub("", s)                         # remove control characters
    s = WS_RE.sub(" ", s)                          # compress runs of whitespace
    s = MANY_NL_RE.sub("\n\n\n", s)               # cap blank lines
    return s.strip()

df_clean = df.copy()
df_clean["subject"] = df_clean["subject"].apply(normalize_newlines).apply(clean_text_for_llm)
df_clean["body"]    = df_clean["body"].apply(normalize_newlines).apply(clean_text_for_llm)

# Build the prompt-friendly email_text with clear section headers
# This is exactly what the model sees at both fine-tuning and inference time
df_clean["email_text"] = (
    "=== EMAIL SUBJECT ===\n" + df_clean["subject"]
    + "\n\n=== EMAIL BODY ===\n" + df_clean["body"]
)

print("Cleaning complete. Shape:", df_clean.shape)

## Step 4 — Deduplicate

In [ ]:
def stable_hash(s: str) -> str:
    """SHA-1 hash of email_text for exact deduplication."""
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

df_clean["text_hash"] = df_clean["email_text"].apply(stable_hash)

before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["text_hash"]).reset_index(drop=True)
after = len(df_clean)

print(f"Before dedup: {before} | After: {after} | Removed: {before - after}")

## Step 5 — Token Length Filtering

Gemma-2-2B-IT has a max context length. We use its tokenizer to measure every email and drop those that exceed 2048 tokens, as they cannot be annotated or trained on reliably.

In [ ]:
from transformers import AutoTokenizer

MAX_TOKENS = 2048  # Gemma context window limit for this task

# Load Gemma tokenizer (requires HuggingFace access)
# Falls back to DeBERTa tokenizer if Gemma is gated
try:
    tok = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
    print("Loaded: google/gemma-2-2b-it tokenizer")
except Exception:
    print("Falling back to microsoft/deberta-v3-base tokenizer")
    tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

In [ ]:
def batched_token_lens(texts, tokenizer, batch_size=256):
    """Tokenize in batches and return token count per text.

    Batching avoids memory issues when processing thousands of emails at once.
    add_special_tokens=False measures only the raw content length.
    """
    lens = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False
        )
        lens.extend([len(ids) for ids in enc["input_ids"]])
    return np.array(lens, dtype=np.int32)

# Compute token length for all cleaned emails
df_clean["token_len"] = batched_token_lens(df_clean["email_text"].tolist(), tok)

# Keep only emails within the 2048-token limit
df_pool = df_clean[df_clean["token_len"] <= MAX_TOKENS].reset_index(drop=True)

print(f"Eligible emails (within {MAX_TOKENS} tokens): {len(df_pool)}")
print(f"Dropped for exceeding token limit:            {len(df_clean) - len(df_pool)}")
print("\nLabel distribution in eligible pool:")
print(df_pool["label"].value_counts())

## Step 6 — Sample 946 Emails from the Cleaned Pool

In [ ]:
# Sample 946 emails from the eligible pool
# random_state=42 makes this fully reproducible
N_KAGGLE = 946

df_kaggle_sample = (
    df_pool
    .sample(n=N_KAGGLE, random_state=RANDOM_SEED)
    [["email_text", "label"]]
    .reset_index(drop=True)
)

print(f"Kaggle sample: {df_kaggle_sample.shape}")
print(df_kaggle_sample["label"].value_counts())

## Step 7 — Define the 50 Hand-Crafted Emails

To cover phishing typologies underrepresented in CEAS_08 (executive impersonation, cloud platform spoofing, academic fraud, immigration scams) and add clean legitimate examples, we manually authored 50 additional emails.

- **5 safe (label=0):** Job alerts, newsletter, meeting notes
- **45 phishing (label=1):** Wire fraud, MFA reset, payroll scam, visa compliance, AWS/GitHub/cloud spoofing, grant fraud, CEO fraud, and more

In [ ]:
# Each entry has subject, body, and label (0=safe, 1=phishing)
crafted_emails = [
    # SAFE EMAILS (label=0)
    {"subject": "Qwen 3.5 Plus, Manus Agents, inference economics",
     "body": "TLDR AI 2026-02-17\n\nQwen3.5-397B-A17B is the first model in the Qwen3.5 series. It uses a hybrid architecture fusing linear attention with sparse mixture-of-experts. Manus Agents is a new way to access Manus directly inside messaging apps. Telegram is currently the only supported app.",
     "label": 0},
    {"subject": "New jobs posted from jobs.paccar.com",
     "body": "You are receiving this email because you joined the PACCAR Talent Community.\n\nYour Job Agent matched:\nSummer 2026 Intern | Field Service Engineer - Kirkland, WA\nSummer 2026 Intern | Materials Planner - Kirkland, WA\nOperations Management Intern - Chillicothe, OH\n\nUpdate your preferences or manage job alerts from your profile.",
     "label": 0},
    {"subject": "New jobs at DISNEY",
     "body": "Check out the latest postings that match your interests:\n\nEvent Services Intern, Summer/Fall 2026 - Walt Disney World Resort, Lake Buena Vista, FL\nInternational Content Strategy Intern - Disney Entertainment Television, Burbank, CA\nABC News Insights and Analytics Intern - New York, NY\n\nManage My Alerts | Unsubscribe",
     "label": 0},
    {"subject": "We found Keysight Opportunities matching your saved job search!",
     "body": "Hi Rishabh,\n\nThe following job matched your alert settings:\nInternship / Working Student - EV Charging and Grid Edge Technologies\nBochum, Nordrhein-Westfalen, Germany\n\nView All Opportunities | Manage Job Alerts\nThank you for your interest in Keysight careers!",
     "label": 0},
    {"subject": "Team Meeting Notes and Next Steps - Q1 Planning",
     "body": "Hi Team,\n\nThank you all for participating in yesterday Q1 planning meeting.\n\nKey takeaways:\n1. Product team will finalize the feature specification draft by next Wednesday.\n2. Engineering will complete the initial architecture review before sprint kickoff.\n3. Finance will circulate the updated budget sheet internally.\n\nPlease continue using our standard internal collaboration tools.\n\nBest,\nSamantha\nProgram Manager",
     "label": 0},

    # PHISHING EMAILS (label=1)
    # Wire / Financial Fraud
    {"subject": "Action Required: Updated Wire Instructions for Q4 Vendor Payment",
     "body": "From: Daniel Harper (CFO) <daniel.harper@financcorp-support.com>\n\nHi Team,\n\nDue to an internal banking transition, our logistics vendor has issued revised remittance instructions for today's transfer.\n\nPlease review the updated payment details in the secure finance portal:\nhttps://secure-financcorp-payments.com/vendor-update\n\nThis update is confidential. Kindly prioritize this before 3:00 PM.\n\nDaniel",
     "label": 1},
    # MFA Reset
    {"subject": "Immediate MFA Reset Confirmation Needed",
     "body": "From: IT Service Desk <support@micr0soft-support365.com>\n\nDear User,\n\nWe detected unusual authentication attempts on your Microsoft 365 account. MFA has been temporarily restricted.\n\nTo avoid mailbox suspension, please re-validate your session:\nhttps://micr0soft-support365.com/session/validate\n\nThis link expires in 45 minutes.\n\nIT Security Operations",
     "label": 1},
    # Payroll
    {"subject": "Payroll Update Required Before Processing Cycle",
     "body": "From: Angela Morris (HR Director) <angela.morris@hr-payroll-update.com>\n\nHello,\n\nDuring our payroll compliance audit, your direct deposit profile was flagged for reconfirmation.\n\nPlease verify your details here:\nhttps://employee-payroll-secure.net/update\n\nRequired before Friday at 5 PM.\n\nAngela",
     "label": 1},
    # Cloud Platform
    {"subject": "AWS Billing Notice: Account Suspension Pending",
     "body": "From: AWS Billing <billing@aws-secure-alert.com>\n\nDear Customer,\n\nOur system shows an unresolved billing discrepancy. Accounts with pending balances may be restricted.\n\nPlease confirm your payment method:\nhttps://aws-billing-verification.net/login\n\nReview within 24 hours.\n\nAWS Billing Team",
     "label": 1},
    # CEO Fraud
    {"subject": "Confidential: CEO Request - Urgent Transfer",
     "body": "From: Michael Reynolds <m.reynolds@executive-corp-mail.com>\n\nHi,\n\nI need assistance with a confidential transaction related to a private acquisition. A preliminary wire must be issued today.\n\nDetails here:\nhttps://executive-docs-secure.com/escrow-details\n\nPlease confirm once complete.\n\nMichael",
     "label": 1},
    # Visa
    {"subject": "Visa Compliance Notice: Immediate Document Review",
     "body": "From: Immigration Desk <support@us-visa-update.org>\n\nDear Student,\n\nYour SEVIS record indicates missing documentation for visa compliance renewal.\n\nUpload confirmation here:\nhttps://visa-status-secure.org/verify\n\nComplete within 48 hours.\n\nImmigration Services",
     "label": 1},
    # Grant
    {"subject": "Research Grant Disbursement Update",
     "body": "From: Grants Office <admin@research-funding-update.com>\n\nDear Investigator,\n\nYour grant allocation requires updated banking confirmation for the next installment release.\n\nSecure confirmation portal:\nhttps://research-grant-confirm.net/login\n\nComplete today to avoid funding delays.\n\nGrants Administration",
     "label": 1},
    {"subject": "Conference Reimbursement Pending Approval", "body": "From: Finance Team <finance@conference-expense-portal.com>\n\nYour conference reimbursement request requires account verification.\n\nValidate details:\nhttps://conference-expense-portal.com/verify\n\nSubmit within 24 hours.\n\nFinance Office", "label": 1},
    {"subject": "GitHub Security Alert: Repository Access Review", "body": "From: GitHub Support <security@github-secure-alert.com>\n\nWe detected a new OAuth integration accessing your repositories.\n\nConfirm activity here:\nhttps://github-security-review.net/session\n\nFailure to review may result in temporary access lock.\n\nGitHub Security", "label": 1},
    {"subject": "Cloud Storage Suspension Warning", "body": "From: Cloud Admin <admin@goo-glecloud-support.com>\n\nYour cloud storage quota exceeded the verified threshold.\n\nUpdate here:\nhttps://accounts.goo-glecloud.com/verify\n\nAvoid data suspension by responding promptly.\n\nCloud Operations", "label": 1},
    {"subject": "Vendor Payment Account Change Notice", "body": "From: Accounts Payable <ap@vendor-secure-change.com>\n\nDue to internal banking restructuring, our remittance account has changed.\n\nUpdated details:\nhttps://vendor-secure-change.com/update-info\n\nConfirm before processing next invoice.\n\nAccounts Team", "label": 1},
    {"subject": "HR Policy Acknowledgment Required", "body": "From: HR Compliance <policy@corp-hr-verify.com>\n\nAnnual compliance acknowledgment is pending.\n\nSign here:\nhttps://corp-hr-verify.com/acknowledge\n\nMandatory completion required.\n\nHR Department", "label": 1},
    {"subject": "Microsoft 365 Storage Lock Alert", "body": "From: Admin Center <notice@micr0soft-tenant-support.com>\n\nYour mailbox storage will be locked due to policy update.\n\nValidate account:\nhttps://micr0soft-tenant-support.com/review\n\nRespond promptly.\n\nAdmin Team", "label": 1},
    {"subject": "Travel Expense Adjustment Confirmation", "body": "From: Travel Desk <travel@reimburse-secure.com>\n\nAn adjustment was made to your travel reimbursement.\n\nReview securely:\nhttps://reimburse-secure.com/check\n\nConfirm within 24 hours.\n\nTravel Finance", "label": 1},
    {"subject": "Payroll Tax Compliance Reminder", "body": "From: Payroll Tax Office <tax@payroll-tax-update.com>\n\nYour tax withholding certificate requires digital confirmation.\n\nAccess here:\nhttps://payroll-tax-update.com/confirm\n\nDeadline approaching.\n\nPayroll Office", "label": 1},
    {"subject": "AWS IAM Credential Verification", "body": "From: AWS IAM <notice@aws-identity-check.com>\n\nAn API key was recently generated in your account.\n\nVerify session:\nhttps://aws-identity-check.com/validate\n\nFailure to confirm may revoke credentials.\n\nAWS Security", "label": 1},
    {"subject": "University Email Quota Warning", "body": "From: IT Services <support@portal-tamu-secure.net>\n\nYour mailbox is nearing capacity under new storage guidelines.\n\nConfirm settings:\nhttps://portal-tamu-secure.net/login\n\nRespond to avoid interruption.\n\nIT Helpdesk", "label": 1},
    {"subject": "Invoice Settlement Reminder", "body": "From: Finance Dept <billing@corp-invoice-review.com>\n\nInvoice #4831 remains unsettled.\n\nView copy:\nhttps://corp-invoice-review.com/access\n\nKindly address today.\n\nFinance", "label": 1},
    {"subject": "MFA Deactivation Notice", "body": "From: Security Admin <admin@secure-mfa-check.com>\n\nMFA settings were altered.\n\nReconfirm here:\nhttps://secure-mfa-check.com/session\n\nPrevent access lock.\n\nSecurity Team", "label": 1},
    {"subject": "Grant Collaboration Invitation", "body": "From: Dr. Helen Porter <h.porter@academic-collab-network.com>\n\nWe invite you to join a funded cross-institutional initiative. Participation requires profile confirmation.\n\nAccess here:\nhttps://academic-collab-network.com/accept\n\nLimited response window.\n\nHelen", "label": 1},
    {"subject": "Payroll Direct Deposit Verification", "body": "From: HR Payroll <payroll@directdeposit-update.com>\n\nYour direct deposit file requires reconfirmation.\n\nSecure portal:\nhttps://directdeposit-update.com/login\n\nComplete promptly.\n\nPayroll Team", "label": 1},
    {"subject": "CEO Travel Reimbursement Urgent", "body": "From: Executive Office <office@ceo-securemail.com>\n\nPlease process reimbursement attached via secure link.\n\nhttps://ceo-securemail.com/document\n\nSensitive matter, confirm completion.\n\nExecutive Office", "label": 1},
    {"subject": "Security Audit: Account Confirmation Required", "body": "From: Compliance <audit@secure-compliance365.com>\n\nDuring routine audit, your profile requires validation.\n\nReview here:\nhttps://secure-compliance365.com/verify\n\nRespond today.\n\nCompliance Team", "label": 1},
    {"subject": "Cloud Drive Sharing Alert", "body": "From: Drive Admin <alert@goo-gle-drive-alert.com>\n\nNew file share detected.\n\nReview activity:\nhttps://goo-gle-drive-alert.com/check\n\nConfirm legitimacy.\n\nDrive Security", "label": 1},
    {"subject": "Vendor Contract Renewal Notice", "body": "From: Procurement <renewal@vendor-contract-secure.com>\n\nContract renewal pending updated payment instructions.\n\nView securely:\nhttps://vendor-contract-secure.com/review\n\nConfirm before renewal.\n\nProcurement", "label": 1},
    {"subject": "Immigration Filing Fee Confirmation", "body": "From: Case Officer <support@visa-fee-secure.org>\n\nYour filing fee confirmation is incomplete.\n\nSubmit here:\nhttps://visa-fee-secure.org/confirm\n\nDeadline within 48 hours.\n\nImmigration Desk", "label": 1},
    {"subject": "GitHub Repository Suspension Notice", "body": "From: GitHub Admin <notice@github-support-alert.com>\n\nPolicy violation review requires credential confirmation.\n\nAccess here:\nhttps://github-support-alert.com/verify\n\nAvoid repository lock.\n\nGitHub Support", "label": 1},
    {"subject": "HR Benefits Enrollment Update", "body": "From: Benefits Team <benefits@corp-benefits-update.com>\n\nEnrollment window closing.\n\nConfirm benefits profile:\nhttps://corp-benefits-update.com/login\n\nMandatory action.\n\nHR Benefits", "label": 1},
    {"subject": "AWS Root Account Verification", "body": "From: AWS Trust and Safety <alert@aws-root-secure.com>\n\nRoot login attempt detected.\n\nValidate here:\nhttps://aws-root-secure.com/auth\n\nFailure to confirm may suspend account.\n\nAWS Security", "label": 1},
    {"subject": "Conference Travel Grant Confirmation", "body": "From: Grants Coordinator <coord@conference-grant-secure.com>\n\nYour travel award requires bank detail confirmation.\n\nSecure form:\nhttps://conference-grant-secure.com/update\n\nRespond promptly.\n\nCoordinator", "label": 1},
    {"subject": "Finance System Upgrade - Login Required", "body": "From: System Admin <admin@finance-upgrade365.com>\n\nSystem upgrade requires credential reconfirmation.\n\nLogin here:\nhttps://finance-upgrade365.com/auth\n\nMandatory validation.\n\nIT Systems", "label": 1},
    {"subject": "Research Funding Compliance Check", "body": "From: Funding Authority <admin@research-compliance-update.com>\n\nCompliance verification pending.\n\nSubmit confirmation:\nhttps://research-compliance-update.com/verify\n\nAction required.\n\nFunding Office", "label": 1},
    {"subject": "Microsoft Teams Account Review", "body": "From: Teams Security <alert@micr0soft-teams-support.com>\n\nSession irregularity detected.\n\nReauthenticate here:\nhttps://micr0soft-teams-support.com/validate\n\nPrevent access suspension.\n\nTeams Admin", "label": 1},
    {"subject": "Cloud Account Compliance Alert", "body": "From: Cloud Security <security@cloud-tenant-check.com>\n\nPolicy violation risk detected.\n\nReview:\nhttps://cloud-tenant-check.com/login\n\nImmediate attention needed.\n\nSecurity", "label": 1},
    {"subject": "University Grant Installment Release", "body": "From: Grants Admin <admin@university-grant-secure.com>\n\nNext installment pending profile validation.\n\nConfirm:\nhttps://university-grant-secure.com/portal\n\nAvoid delay.\n\nGrants Team", "label": 1},
    {"subject": "GitHub Billing Discrepancy", "body": "From: GitHub Billing <billing@github-payment-alert.com>\n\nBilling method declined.\n\nUpdate here:\nhttps://github-payment-alert.com/update\n\nAvoid service disruption.\n\nGitHub Billing", "label": 1},
    {"subject": "AWS Security Token Expiry", "body": "From: AWS Security <notice@aws-token-reset.com>\n\nYour security token expires today.\n\nReset session:\nhttps://aws-token-reset.com/renew\n\nConfirm promptly.\n\nAWS Team", "label": 1},
    {"subject": "HR Confidential Compensation Review", "body": "From: HR Executive <executive@hr-confidential-update.com>\n\nConfidential compensation file requires acknowledgment.\n\nAccess here:\nhttps://hr-confidential-update.com/view\n\nDo not forward.\n\nHR Executive", "label": 1},
    {"subject": "Finance Compliance Escalation", "body": "From: Compliance Office <notice@finance-compliance-alert.com>\n\nYour account flagged for compliance review.\n\nValidate here:\nhttps://finance-compliance-alert.com/confirm\n\nRespond today.\n\nCompliance Office", "label": 1},
    {"subject": "Executive NDA Confirmation", "body": "From: Legal Office <legal@nda-secureportal.com>\n\nConfidential NDA requires signature confirmation.\n\nSign here:\nhttps://nda-secureportal.com/review\n\nTime-sensitive.\n\nLegal Department", "label": 1},
    {"subject": "Vendor ACH Update Required", "body": "From: Accounts Dept <accounts@vendor-ach-update.com>\n\nACH remittance update required.\n\nConfirm here:\nhttps://vendor-ach-update.com/form\n\nBefore next cycle.\n\nAccounts", "label": 1},
]

print(f"Total crafted emails: {len(crafted_emails)}")
print(f"  Safe (0): {sum(1 for e in crafted_emails if e['label']==0)}")
print(f"  Phishing (1): {sum(1 for e in crafted_emails if e['label']==1)}")

## Step 8 — Combine into Final 996-Sample Dataset

In [ ]:
def format_email_text(subject: str, body: str) -> str:
    """Apply same email_text format used for Kaggle emails."""
    return (
        "=== EMAIL SUBJECT ===\n" + str(subject)
        + "\n\n=== EMAIL BODY ===\n" + str(body)
    )

# Convert crafted list to DataFrame with same column format as Kaggle sample
df_crafted = pd.DataFrame([
    {"email_text": format_email_text(e["subject"], e["body"]), "label": int(e["label"])}
    for e in crafted_emails
])

# Concatenate: crafted emails come first, then Kaggle sample
# This matches the ordering used in generate_teacher_dataset (extra_emails on top)
df_teacher = pd.concat([df_crafted, df_kaggle_sample], ignore_index=True)

print(f"Final dataset shape: {df_teacher.shape}  (expected 996 rows)")
print(df_teacher["label"].value_counts())

# Save crafted emails separately for audit
df_crafted.to_csv("extra_50_crafted.csv", index=False)
print("Saved: extra_50_crafted.csv")

---
# Part 2 — Teacher Annotation with Gemini 2.5 Pro

For each of the 996 emails, we call **Gemini 2.5 Pro** (the teacher LLM) with a structured prompt.
The teacher generates a JSON object containing:
- `label`: PHISHING or SAFE (matching the true label)
- `explanation`: 2-3 sentence non-technical explanation
- `evidence_snippets`: 2-4 verbatim excerpts from the email body
- `user_advice`: 2-3 short actionable recommendations

Each teacher-annotated record is saved in **OpenAI-style chat JSONL format** so it can be directly used to fine-tune Gemma.

## Step 9 — Set Up Gemini API

In [ ]:
!pip -q install google-generativeai
import google.generativeai as genai

# Prompt for Google API key if not already set in environment
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your GOOGLE_API_KEY: ")

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
print("Gemini API configured.")

## Step 10 — Define the Teacher Instruction Prompt

The teacher prompt is carefully engineered to:
- Lock the `label` field to match the ground truth (no hallucinated labels)
- Extract evidence that is verbatim from the email (character-for-character)
- Keep explanations short and non-technical (for non-expert SME users)
- Return strict JSON with exactly 4 keys and no extra text

In [ ]:
def label_to_str(y: int) -> str:
    """Convert numeric label to string expected by the teacher prompt."""
    return "PHISHING" if int(y) == 1 else "SAFE"

# The teacher instruction injected before every email
# LABEL is filled at call time with the ground truth label
TEACHER_INSTRUCTION = """You are a cybersecurity analyst writing an educational explanation for a non-technical user.
The true label is: {LABEL}.

TASK:
- Write exactly 2-3 short, highly relevant and direct sentences (no more than 80 words total)
  that match the true label.
- Extract 2-4 exact snippets from the email verbatim,
  character-for-character, and place them only inside the "evidence_snippets" array.
  Do not repeat them verbatim inside the explanation text.
- If the evidence is limited or ambiguous, acknowledge the uncertainty
  while still aligning with the true label and providing cautious advice.
- Provide 2-3 short actionable recommendations (each under 20 words).
- Do NOT mention datasets, machine learning, or training.

OUTPUT FORMAT (STRICT):
Return ONLY a single JSON object (no markdown, no backticks, no extra text).
The "label" field must exactly match "{LABEL}".
The JSON must contain exactly these keys and no others:
- "label"
- "explanation"
- "evidence_snippets"
- "user_advice"
"""

def build_teacher_prompt(email_text: str, label: int) -> str:
    """Fill in the true label and append the email body to the teacher instruction."""
    lbl = label_to_str(label)
    return TEACHER_INSTRUCTION.format(LABEL=lbl) + "\n\nEMAIL:\n" + email_text

# Preview what the teacher sees for one sample
sample_prompt = build_teacher_prompt(df_teacher["email_text"].iloc[0], df_teacher["label"].iloc[0])
print(sample_prompt[:600])

## Step 11 — Define the Gemini API Call Function

We call Gemini 2.5 Pro with `response_mime_type="application/json"` to force structured JSON output.
The function retries up to 3 times on parse failures and validates the response schema before returning.

In [ ]:
# The 4 keys that every valid teacher response must contain
REQUIRED_KEYS = {"label", "explanation", "evidence_snippets", "user_advice"}

def call_teacher_gemini(prompt: str, client, max_retries: int = 3) -> dict:
    """Call Gemini 2.5 Pro and return a validated JSON dict.

    Retries up to max_retries times on JSON parse failure or schema violation.
    Uses response_mime_type=application/json to force structured output.

    Args:
        prompt: The full teacher prompt including email text
        client: A genai.GenerativeModel instance
        max_retries: How many times to retry on failure

    Returns:
        dict with keys: label, explanation, evidence_snippets, user_advice

    Raises:
        RuntimeError if all retries are exhausted
    """
    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            # Two-turn message: first a system reminder, then the actual prompt
            messages = [
                {"role": "user", "parts": ["Return ONLY valid JSON. Follow the STRICT output format exactly."]},
                {"role": "user", "parts": [prompt]},
            ]

            resp = client.generate_content(
                messages,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.2,                          # low temperature for consistent structured output
                    response_mime_type="application/json",   # force JSON response from Gemini
                ),
            )

            data = json.loads(resp.text)

            # Validate all required keys are present
            if not REQUIRED_KEYS.issubset(set(data.keys())):
                raise ValueError(f"Missing keys. Got {set(data.keys())}, need {REQUIRED_KEYS}")

            # Validate evidence_snippets is a list with 2-4 items
            if (not isinstance(data["evidence_snippets"], list)) or not (2 <= len(data["evidence_snippets"]) <= 4):
                raise ValueError("evidence_snippets must be a list with 2-4 items")

            # Validate label is one of the two expected values
            if data["label"] not in ["PHISHING", "SAFE"]:
                raise ValueError(f"Invalid label returned: {data['label']}")

            # Return only the 4 required keys, stripping any extra fields
            return {k: data[k] for k in REQUIRED_KEYS}

        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
            time.sleep(1.5 * attempt)  # exponential backoff between retries

    raise RuntimeError(f"Failed after {max_retries} tries. Last error: {last_err}")

## Step 12 — Run Teacher Annotation (Generate 996 Explanations)

We iterate over all 996 emails and call Gemini 2.5 Pro for each one.
Each annotated email is saved immediately to a JSONL file in **OpenAI-style chat format**:

```json
{
  "messages": [
    {"role": "user",      "content": "Analyze this email...\n\nEMAIL:\n..."},
    {"role": "assistant", "content": "{\"label\": \"PHISHING\", \"explanation\": \"...\", ...}"}
  ]
}
```

This format is directly compatible with Gemma fine-tuning via the `apply_chat_template` method.  
We append to the file after each successful record so progress is not lost on failure.

In [ ]:
# Output JSONL file that will be fed to the fine-tuning pipeline
OUT_PATH = "teacher_gemini_explanations_chat.jsonl"

def append_jsonl(record: dict, path: str):
    """Append a single JSON record to a JSONL file.

    We write after each record (not batched) so partial progress is preserved
    if the run is interrupted or a quota error occurs.
    """
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def generate_teacher_dataset(
    df_teacher: pd.DataFrame,
    n: int = 950,
    model: str = "gemini-2.5-pro",
    extra_emails: list = None,
    random_state: int = 42,
    sleep_s: float = 0.8
):
    """Call Gemini 2.5 Pro for every email and save annotated records to JSONL.

    Args:
        df_teacher: Cleaned email pool (columns: email_text, label)
        n: Number of emails to sample from df_teacher (here 950, which after
           evidence filtering yields ~946 usable samples)
        model: Gemini model name to use as teacher
        extra_emails: List of hand-crafted email dicts (subject, body, label)
                      These are prepended before the Kaggle sample
        random_state: Seed for reproducible sampling from df_teacher
        sleep_s: Seconds to sleep between API calls to avoid rate limiting
    """
    # Create a single Gemini client for the full run (avoid reinitializing per call)
    client = genai.GenerativeModel(model)

    # Randomly sample n emails from the Kaggle pool
    subset = (
        df_teacher
        .sample(n=n, random_state=random_state)
        [["email_text", "label"]]
        .reset_index(drop=True)
    )

    # Prepend extra hand-crafted emails if provided
    # These come first so they are always included regardless of Kaggle sampling
    if extra_emails:
        extra_rows = []
        for e in extra_emails:
            if not all(k in e for k in ("subject", "body", "label")):
                raise ValueError("Each extra email must have: subject, body, label")
            extra_rows.append({
                "email_text": format_email_text(e["subject"], e["body"]),
                "label": int(e["label"])
            })
        extra_df = pd.DataFrame(extra_rows)
        subset = pd.concat([extra_df, subset], ignore_index=True)
        print(f"Prepended {len(extra_df)} crafted emails. Total to annotate: {len(subset)}")
    else:
        print(f"Total to annotate: {len(subset)}")

    success = 0
    failures = 0

    for idx, row in subset.iterrows():
        try:
            # Build the teacher prompt with ground truth label
            prompt = build_teacher_prompt(row["email_text"], int(row["label"]))

            # Call Gemini and get structured JSON response
            result = call_teacher_gemini(prompt, client)

            # The assistant content is the structured JSON as a string
            assistant_obj = {
                "label":             result["label"],
                "explanation":       result["explanation"],
                "evidence_snippets": result["evidence_snippets"],
                "user_advice":       result["user_advice"],
            }

            # Format as chat-style JSONL record for Gemma fine-tuning
            # The user turn is the email analysis request
            # The assistant turn is the structured JSON explanation
            rec = {
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            "Analyze this email and determine if it is phishing or safe. "
                            "Explain your reasoning and give educational advice.\n\nEMAIL:\n"
                            + row["email_text"]
                        ),
                    },
                    {
                        "role": "assistant",
                        "content": json.dumps(assistant_obj, ensure_ascii=False),
                    },
                ]
            }

            # Write immediately so we don't lose progress on interruption
            append_jsonl(rec, OUT_PATH)
            success += 1

            if success % 50 == 0:
                print(f"Progress: {success} records saved...")

            # Polite sleep to stay within Gemini API rate limits
            time.sleep(sleep_s)

        except Exception as e:
            failures += 1
            print(f"Failed at row {idx}: {e}")

    print(f"\nDone. Success: {success} | Failures: {failures}")
    print(f"Output written to: {OUT_PATH}")

In [ ]:
# Run the full teacher annotation
# n=950 from Kaggle pool + 50 crafted emails = ~1000 attempts
# After evidence filtering in the next steps, this yields ~996 clean samples
generate_teacher_dataset(
    df_teacher,
    n=950,
    model="gemini-2.5-pro",
    extra_emails=crafted_emails,
    random_state=RANDOM_SEED,
    sleep_s=0.8
)

## Step 13 — Validate the Raw JSONL Output

Before filtering, we run quick checks to measure:
- How many records parsed successfully as JSON
- Label distribution across the annotated set
- Evidence snippet mismatch rate (snippets not found in the original email)

In [ ]:
# Count total records and JSON parse failures
bad_json = 0
total = 0

with open(OUT_PATH, "r") as f:
    for line in f:
        total += 1
        try:
            rec = json.loads(line)
            json.loads(rec["messages"][1]["content"])  # verify assistant JSON is also valid
        except Exception:
            bad_json += 1

print(f"Total records: {total}")
print(f"JSON parse failures: {bad_json}")

In [ ]:
# Check label distribution returned by the teacher
# Should closely match the original dataset labels
rows = []
with open(OUT_PATH) as f:
    for line in f:
        rec = json.loads(line)
        assistant = json.loads(rec["messages"][1]["content"])
        rows.append({"assistant_label": assistant["label"]})

df_check = pd.DataFrame(rows)
print("Teacher label distribution:")
print(df_check["assistant_label"].value_counts(normalize=True))

In [ ]:
# Measure evidence snippet mismatch rate BEFORE filtering
# A snippet is mismatched if it does not appear verbatim in the email text
# High mismatch = teacher hallucinated evidence (must be filtered out)
bad_evidence = 0
total_snippets = 0

with open(OUT_PATH) as f:
    for line in f:
        rec = json.loads(line)
        email = rec["messages"][0]["content"]
        assistant = json.loads(rec["messages"][1]["content"])
        for snippet in assistant["evidence_snippets"]:
            total_snippets += 1
            if snippet not in email:
                bad_evidence += 1

print(f"Evidence mismatch rate (before filtering): {bad_evidence / total_snippets:.3f}")
print(f"  ({bad_evidence} out of {total_snippets} snippets not found verbatim in email)")

## Step 14 — Filter Hallucinated Evidence Snippets

Some evidence snippets generated by Gemini do not appear verbatim in the original email.
These are hallucinations and must be removed before the data is used for fine-tuning.

We apply light unicode normalization before matching (to handle quote/dash variants),
and drop the entire record if fewer than 2 valid snippets remain after filtering.

In [ ]:
# Unicode normalization for fuzzy matching of quotes and dashes
def norm(s: str) -> str:
    """Normalize unicode quotes/dashes and collapse whitespace for snippet matching."""
    s = s.replace("\u2019", "'").replace("\u2018", "'")   # smart quotes
    s = s.replace("\u201c", '"').replace("\u201d", '"')    # curly double quotes
    s = s.replace("\u2013", "-").replace("\u2014", "-")    # en/em dashes
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def filter_valid_snippets(email: str, snippets: list, min_chars: int = 10) -> list:
    """Keep only snippets that appear in the email and are long enough to be meaningful.

    Args:
        email: The full email text (user turn content)
        snippets: List of evidence snippet strings from the teacher
        min_chars: Minimum character length for a valid snippet

    Returns:
        List of valid snippets (subset of input)
    """
    email_n = norm(email)
    valid = []
    for sn in (snippets or []):
        sn_norm = norm(sn).strip()
        # Drop trivially short snippets like "click here"
        if len(sn_norm) < min_chars:
            continue
        # Keep only if the snippet can be found in the normalized email
        if sn_norm in email_n:
            valid.append(sn)
    return valid

# Filter constants
MIN_SNIPPETS = 2   # drop whole record if fewer than 2 valid snippets remain
MAX_SNIPPETS = 4   # cap at 4 snippets per record

IN_PATH  = "teacher_gemini_explanations_chat.jsonl"
OUT_FILTERED = "teacher_gemini_explanations_chat.filtered.jsonl"

written = 0
dropped = 0

with open(IN_PATH, "r", encoding="utf-8") as fin, open(OUT_FILTERED, "w", encoding="utf-8") as fout:
    for line in fin:
        rec = json.loads(line)
        email = rec["messages"][0]["content"]
        assistant = json.loads(rec["messages"][1]["content"])

        # Filter snippets: keep only those grounded in the actual email
        valid_snippets = filter_valid_snippets(email, assistant.get("evidence_snippets", []))
        valid_snippets = valid_snippets[:MAX_SNIPPETS]  # cap at 4

        # Drop the entire record if not enough valid snippets remain
        if len(valid_snippets) < MIN_SNIPPETS:
            dropped += 1
            continue

        # Update the record with the filtered snippets
        assistant["evidence_snippets"] = valid_snippets
        rec["messages"][1]["content"] = json.dumps(assistant, ensure_ascii=False)

        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        written += 1

print(f"Written (passed filter): {written}")
print(f"Dropped (too few valid snippets): {dropped}")
print(f"Output: {OUT_FILTERED}")

## Step 15 — Produce the Final Clean JSONL

Remove any internal metadata fields added during the filter step, leaving only the 4 required assistant keys.
This is the final file used for Gemma fine-tuning.

In [ ]:
FINAL_JSONL = "teacher_gemini_explanations_chat.filtered.clean.jsonl"

# Strip any internal metadata fields that were added during validation
META_FIELDS = {"evidence_snippets_validated", "evidence_validation"}

with open(OUT_FILTERED, "r", encoding="utf-8") as fin, open(FINAL_JSONL, "w", encoding="utf-8") as fout:
    for line in fin:
        rec = json.loads(line)
        a = json.loads(rec["messages"][1]["content"])

        # Remove metadata keys if present
        for field in META_FIELDS:
            a.pop(field, None)

        rec["messages"][1]["content"] = json.dumps(a, ensure_ascii=False)
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Final record count
final_count = sum(1 for _ in open(FINAL_JSONL))
print(f"Final clean JSONL: {FINAL_JSONL}")
print(f"Total records ready for fine-tuning: {final_count}")

In [ ]:
# Final sanity check: verify mismatch rate is now 0 in the clean file
bad_evidence = 0
total_snippets = 0

with open(FINAL_JSONL) as f:
    for line in f:
        rec = json.loads(line)
        email = rec["messages"][0]["content"]
        assistant = json.loads(rec["messages"][1]["content"])
        for snippet in assistant.get("evidence_snippets", []):
            total_snippets += 1
            if norm(snippet) not in norm(email):
                bad_evidence += 1

print(f"Evidence mismatch rate (after filtering): {bad_evidence / max(total_snippets, 1):.4f}")
print(f"Total snippets in final dataset: {total_snippets}")

---
## Summary

### Part 1: Dataset Construction

| Step | Action | Result |
|------|--------|--------|
| 1 | Download CEAS_08.csv from Kaggle | Raw CSV |
| 2 | Load and select subject, body, label columns | `df` |
| 3 | Normalize newlines, build email_text, clean text | `df_clean` |
| 4 | SHA-1 deduplication | `df_clean` |
| 5 | Filter to max 2048 tokens (Gemma limit) | `df_pool` |
| 6 | Sample 946 emails (seed=42) | `df_kaggle_sample` |
| 7 | Define 50 hand-crafted emails | `crafted_emails` |
| 8 | Concatenate 50 + 946 = 996 rows | `df_teacher` |

### Part 2: Teacher Annotation

| Step | Action | Result |
|------|--------|--------|
| 9 | Configure Gemini 2.5 Pro API | Authenticated client |
| 10 | Define teacher instruction prompt | `TEACHER_INSTRUCTION` |
| 11 | Define Gemini call with validation and retry logic | `call_teacher_gemini` |
| 12 | Annotate all 996 emails, save as chat JSONL | `teacher_gemini_explanations_chat.jsonl` |
| 13 | Validate label distribution and snippet mismatch rate | Quality report |
| 14 | Filter hallucinated evidence snippets | `.filtered.jsonl` |
| 15 | Strip metadata, produce final clean JSONL | `.filtered.clean.jsonl` (ready for fine-tuning) |